# 01 · Баланс правил v1.0

Два вопроса из `DEV_PLAN` (спринт 3):

1. **Воспроизводит ли v1.0 коллапс экологии из Смольного?** В реальной партии экология упала до 0 к 4-му раунду, 7 городов уничтожены за один раунд.
2. **Какой бот чаще побеждает?** Если одна стратегия доминирует, правила её вознаграждают.

Плюс четыре эксперимента с ценами (`rules/experiments/`) через `arena.sim.compare` — парные партии на одних зёрнах.

Выводы — в `docs/rules/v1-balance-notes.md`. Ноутбук пересчитывает их с нуля; на 12 ядрах ~30 секунд.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import polars as pl
import matplotlib.pyplot as plt
from statistics import mean
from collections import Counter

from arena.sim import specs_for, run_batch, summarize
from arena.sim.compare import compare

N = 1000
WORKERS = None  # все ядра

## 1. Базовая линия: смешанные составы

In [ ]:
base = {}
for scenario in ("smolny", "equal"):
    metrics, _ = run_batch(specs_for("v1.0", scenario, range(N), "mixed"), workers=WORKERS)
    base[scenario] = summarize(metrics)
    print(scenario)
    for k, v in base[scenario].items():
        print(f"  {k:<26} {v}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
bots = sorted(base["smolny"]["win_rate_by_bot"])
for i, scenario in enumerate(("smolny", "equal")):
    ax.bar([x + i * 0.4 for x in range(len(bots))],
           [base[scenario]["win_rate_by_bot"][b] for b in bots], width=0.4, label=scenario)
ax.axhline(0.2, ls="--", c="grey", lw=1, label="справедливые 20 %")
ax.set_xticks([x + 0.2 for x in range(len(bots))], bots)
ax.set_ylabel("доля побед"); ax.set_title("Кто побеждает при смешанных составах, v1.0"); ax.legend()
plt.show()

## 2. Экология по раундам: есть ли коллапс Смольного?

In [ ]:
metrics, logs = run_batch(specs_for("v1.0", "smolny", range(300), "mixed"), keep_logs=True)
rounds = range(1, 7)
eco = [[log.rounds[r - 1].state_after.ecology for log in logs] for r in rounds]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot(eco, positions=list(rounds), widths=0.5)
ax.plot([1, 2, 3, 4], [100, 95, 95, 0], "r--o", label="Смольный (реальная партия, оценка)")
ax.set_xlabel("раунд"); ax.set_ylabel("мировая экология, %"); ax.legend()
ax.set_title("v1.0, смешанные боты, 300 партий")
plt.show()
for r in rounds:
    print(f"после раунда {r}: среднее {mean(eco[r-1]):5.1f}, минимум {min(eco[r-1]):3d}, "
          f"доля < 30 %: {sum(v < 30 for v in eco[r-1]) / len(logs):.2f}")
dead = Counter(sum(c.destroyed for co in log.final_state.countries for c in co.cities) for log in logs)
print("уничтоженных городов к концу:", sorted(dead.items()))

В реальной партии всё было хуже, чем у ботов: там 4 из 5 стран вооружились и в один раунд снесли 7 городов. Боты в смешанном составе доходят до 0 лишь в 1 % партий. Что даёт «все агрессоры»:

In [ ]:
for lineup in ("aggressor", "economist,economist,economist,economist,aggressor"):
    m, _ = run_batch(specs_for("v1.0", "smolny", range(100), lineup), workers=WORKERS)
    s = summarize(m)
    print(lineup, "→ экология", s["mean_final_ecology"], "коллапс", s["ecology_collapse_share"],
          "на раунде", s["mean_collapse_round"], "городов", s["mean_cities_destroyed"])

## 3. Эксперименты с ценами (парные партии)

In [ ]:
experiments = ["cheap-eco", "costly-bomb", "dirty-strike", "cheap-shield"]
results = {}
for exp in experiments:
    results[exp] = compare("v1.0", f"experiments/v1.0-{exp}", range(N), scenario="smolny", workers=WORKERS)
    print("\n" + "=" * 70); print(results[exp].report())

In [ ]:
rows = []
for exp, c in results.items():
    rows.append({"experiment": exp,
                 "Δ ecology": c.paired_diffs["final_ecology"],
                 "Δ strikes": c.paired_diffs["strikes"],
                 "Δ cities destroyed": c.paired_diffs["cities_destroyed"],
                 "aggressor win A": c.summary_a["win_rate_by_bot"]["aggressor"],
                 "aggressor win B": c.summary_b["win_rate_by_bot"]["aggressor"],
                 "winner changed": c.winner_changed_share})
pl.DataFrame(rows)

## 4. Что читать дальше

`docs/rules/v1-balance-notes.md` — выводы и что из этого следует для v2. Сырые метрики каждой партии лежат в `data/sim/*.parquet`; например:

In [ ]:
df = pl.read_parquet(ROOT / "data" / "sim" / "v1.0_smolny_mixed_1000.parquet") if (ROOT / "data" / "sim" / "v1.0_smolny_mixed_1000.parquet").exists() else None
df.head() if df is not None else "запустите just sim 1000"
